# UD7.02. Cuatro maneras de vaciar el buzón

**Módulo 5073 · Programación de Inteligencia Artificial · Curso 2026/27**
Bloques 3 y 4 de los apuntes · Criterio **4.c**

---

El cuaderno anterior midió lo que hay: 18.805 euros al año, un 86,8 % de acierto y 62
garantías mal encaminadas. Este mide **cuatro maneras distintas de quitarle ese trabajo
a la persona**, todas sobre el mismo buzón, con la misma partición y contra los mismos
dos puntos de referencia.

| Modelo | Qué es |
|---|---|
| Clase mayoritaria | Contestar siempre lo más frecuente. El suelo |
| Reglas por palabras clave | Seis expresiones escritas a mano |
| Modelo entrenado, solo texto | Un clasificador aprendido de los mensajes anteriores |
| Modelo entrenado, texto + contexto | El mismo, con un dato que la empresa ya tenía |

Y un quinto, el **modelo de lenguaje grande**, que aquí no se mide: se le pone precio y
se declara honestamente lo que no sabemos de él.

Al final tienes que poder contestar:

1. ¿Cuánto sube cada modelo respecto del anterior, y cuánto trabajo cuesta esa subida?
2. ¿Dónde se equivoca cada uno? Porque no se equivocan en los mismos mensajes.
3. ¿Por qué la diferencia entre el tercero y el cuarto **no es tecnológica**?
4. ¿Cuál es el techo de este problema y por qué existe?

In [ ]:
import os
import time
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", None)
SEMILLA = 20262027

# Solo en Colab. En local el fichero ya está en recursos/datos/.
BASE = ("https://raw.githubusercontent.com/RafaSalaEsteve/IABD-PIA-notebooks"
        "/main/UD7/datos/")
os.makedirs("datos", exist_ok=True)
RUTA = os.path.join("datos", "buzon_techstore.csv")
if not os.path.exists(RUTA):
    urllib.request.urlretrieve(BASE + "buzon_techstore.csv", RUTA)
    print("descargado")

buzon = pd.read_csv(RUTA, parse_dates=["fecha_hora"])
print(f"{len(buzon):,} mensajes, de {buzon.fecha_hora.min().date()} "
      f"a {buzon.fecha_hora.max().date()}")

## 1. La partición, y por qué es temporal

El buzón tiene tiempo dentro: los mensajes de enero no se parecen a los de diciembre
—cambia el catálogo, cambian las campañas, en enero llegan las devoluciones de Navidad—.
Si se parte al azar, el modelo entrena con mensajes de diciembre y se evalúa con
mensajes de enero **que están entre medias de los que ha visto**, y el resultado sale
mejor de lo que será en producción.

Es exactamente el mismo problema que la UD3 y la UD5 tenían con los clientes de
TechStore. Aquí se entrena con los nueve primeros meses y se prueba con los tres
últimos, que es como funcionará de verdad: se entrena con el pasado y se usa en el
futuro.

Al final del cuaderno mediremos **cuánto sube el resultado haciendo la trampa**, para
que la diferencia no sea un artículo de fe.

In [ ]:
CORTE = pd.Timestamp("2024-10-01")
entrena = buzon[buzon["fecha_hora"] < CORTE].copy()
prueba = buzon[buzon["fecha_hora"] >= CORTE].copy()
verdad = prueba["categoria"].to_numpy()

print(f"entrenamiento: {len(entrena):>5} mensajes (hasta {CORTE.date()})")
print(f"prueba:        {len(prueba):>5} mensajes")
print()
print("Reparto de categorías en las dos partes:")
comparacion = pd.DataFrame({
    "entrena": entrena["categoria"].value_counts(normalize=True),
    "prueba": prueba["categoria"].value_counts(normalize=True),
}).sort_values("entrena", ascending=False)
print((comparacion * 100).round(1))

## 2. Modelo 0: la clase mayoritaria

El suelo. Contestar siempre «envío», que es lo más frecuente, sin mirar el mensaje.

Ningún modelo se informa sin esto. Es la misma norma de la UD4 y de la UD5: **una
exactitud sin punto de referencia no significa nada.**

In [ ]:
COSTE_ERROR = {"spam": 1.50, "envio": 8.00, "devolucion": 18.00,
               "factura": 25.00, "producto": 35.00, "garantia": 120.00}
PENALIZACION_A_SPAM = 2.5


def coste_de_los_errores(datos, prediccion):
    """Coste de los errores de una predicción, escalado al año entero."""
    real = datos["categoria"].to_numpy()
    fallos = prediccion != real
    coste = sum(COSTE_ERROR[r] * (PENALIZACION_A_SPAM if p == "spam" else 1.0)
                for r, p in zip(real[fallos], prediccion[fallos]))
    return coste / len(datos) * len(buzon)


mayoritaria = entrena["categoria"].value_counts().idxmax()
pred_mayoritaria = np.full(len(prueba), mayoritaria)

resultados = {}


def anota(nombre, prediccion, segundos=None):
    resultados[nombre] = {
        "exactitud": float((prediccion == verdad).mean()),
        "coste_errores": coste_de_los_errores(prueba, prediccion),
        "prediccion": prediccion,
        "ms_por_mensaje": None if segundos is None else segundos / len(prueba) * 1000,
    }
    r = resultados[nombre]
    print(f"{nombre:<28} exactitud {r['exactitud']:>6.1%}   "
          f"errores {r['coste_errores']:>9,.0f} EUR/año")


print(f"La clase mayoritaria es «{mayoritaria}».\n")
anota("clase mayoritaria", pred_mayoritaria)

## 3. Modelo 1: reglas por palabras clave

El modelo más antiguo y el más subestimado. Seis expresiones regulares, en orden de
prioridad: gana la primera que casa.

El orden **no es inocente**. Se ponen primero las categorías cuyo error es más caro y
las que tienen vocabulario más propio; «pedido» va casi al final porque aparece en todos
los mensajes. Escribir estas seis reglas son unas dos horas de trabajo, contando las
pruebas.

In [ ]:
import re

REGLAS = [
    ("spam", r"oferta|descuento|propuesta|posicionamiento|colaboraci|inversi|"
             r"base de datos|confirme sus datos|acceso no autorizado|externalizado|tarifas"),
    ("garantia", r"garant|no enciende|estropea|aver|repar|defectuos|roto|no funciona|falla|bater"),
    ("devolucion", r"devolv|devoluci|devuelv|anular|cambiar|reembols"),
    ("factura", r"factur|nif|cobrad|importe|iva|transferencia|pendiente de pago"),
    ("envio", r"pedido|env[ií]|seguimiento|reparto|entrega|repartidor|llega|paquete"),
    ("producto", r"compatib|stock|existencias|modelo|color|cables|ficha"),
]


def clasifica_por_reglas(texto):
    t = texto.lower()
    for categoria, patron in REGLAS:
        if re.search(patron, t):
            return categoria
    return mayoritaria      # si no casa ninguna, lo más frecuente


inicio = time.perf_counter()
pred_reglas = prueba["texto"].map(clasifica_por_reglas).to_numpy()
segundos_reglas = time.perf_counter() - inicio

anota("reglas por palabras clave", pred_reglas, segundos_reglas)
print(f"\nTiempo: {segundos_reglas * 1000 / len(prueba):.3f} ms por mensaje")
sin_casar = prueba["texto"].map(
    lambda t: not any(re.search(p, t.lower()) for _, p in REGLAS)).mean()
print(f"Mensajes que no casan con ninguna regla: {sin_casar:.1%}")

> **Del 26 % al 72,7 % con seis expresiones regulares y dos horas de trabajo.**
>
> Éste, y no el azar, es el punto de referencia que tiene que batir cualquier modelo
> para justificarse. Una propuesta que compare su modelo contra «no hacer nada» se está
> apuntando un mérito que no es suyo.

Las reglas tienen una propiedad que conviene ver, porque decide dónde se pueden usar:
**son totalmente explicables**. Se puede decir exactamente por qué un mensaje fue a un
sitio. Y tienen la contraria: **fallan de golpe**, sin avisar y sin degradar.

In [ ]:
# El orden de las reglas importa tanto como las reglas. Vamos a medirlo: se prueba el
# mismo conjunto con el orden invertido.
REGLAS_INVERTIDAS = list(reversed(REGLAS))


def clasifica_invertido(texto):
    t = texto.lower()
    for categoria, patron in REGLAS_INVERTIDAS:
        if re.search(patron, t):
            return categoria
    return mayoritaria


pred_invertido = prueba["texto"].map(clasifica_invertido).to_numpy()
print(f"Con el orden original:  {float((pred_reglas == verdad).mean()):.1%}")
print(f"Con el orden invertido: {float((pred_invertido == verdad).mean()):.1%}")
print()
print("Las reglas son las mismas. Lo único que cambia es en qué orden se prueban, y")
print("eso vale más de veinte puntos. Un sistema de reglas no es una lista: es una")
print("lista ORDENADA, y ese orden es conocimiento del negocio que hay que documentar.")

## 4. Modelo 2: un clasificador entrenado con el texto

Ahora un modelo aprendido de los ejemplos. Dos piezas:

- **TF-IDF** convierte cada mensaje en un vector de números: una dimensión por palabra o
  pareja de palabras, con más peso a las palabras raras. Es la representación que la UD3
  llamaba «pasar del texto a la tabla».
- **Regresión logística**, el mismo modelo de una capa que la UD5 construyó a mano.

Lo importante no es la técnica: es que **no hemos escrito ninguna regla**. Las 1.773
decisiones que la persona tomó antes de octubre son el material de partida.

In [ ]:
vectorizador = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_entrena = vectorizador.fit_transform(entrena["texto"])
X_prueba = vectorizador.transform(prueba["texto"])
print(f"Vocabulario aprendido: {len(vectorizador.vocabulary_):,} términos")

inicio = time.perf_counter()
modelo_texto = LogisticRegression(max_iter=2000, C=5).fit(X_entrena, entrena["categoria"])
segundos_entrenamiento = time.perf_counter() - inicio

inicio = time.perf_counter()
pred_texto = modelo_texto.predict(X_prueba)
segundos_texto = time.perf_counter() - inicio

anota("modelo, solo texto", pred_texto, segundos_texto)
print(f"\nEntrenar: {segundos_entrenamiento:.1f} s")
print(f"Clasificar: {segundos_texto * 1000 / len(prueba):.3f} ms por mensaje")

Sube al 90,6 % y ya bate a la persona, que se quedaba en el 85,3 %. Conviene decir por
qué, porque la razón no es que sea más listo:

> **Un modelo no se cansa a las seis de la tarde y aplica el mismo criterio el lunes que
> el viernes.** La ventaja real de una máquina en un proceso repetitivo es la
> consistencia, y no depende de lo sofisticada que sea.

Pero hay un sitio donde este modelo lo hace francamente mal. Vamos a buscarlo.

In [ ]:
grupos = {
    "de frontera": prueba["frontera"] == 1,
    "confusos": prueba["confuso"] == 1,
    "con dos intenciones": prueba["ambiguo"] == 1,
    "nítidos": (prueba["frontera"] == 0) & (prueba["confuso"] == 0) & (prueba["ambiguo"] == 0),
}

print(f"{'grupo':<22} {'n':>5} {'reglas':>9} {'modelo':>9} {'persona':>9}")
for nombre, mascara in grupos.items():
    m = mascara.to_numpy()
    print(f"{nombre:<22} {m.sum():>5} "
          f"{float((pred_reglas[m] == verdad[m]).mean()):>8.0%} "
          f"{float((pred_texto[m] == verdad[m]).mean()):>8.0%} "
          f"{float((prueba['categoria_asignada'].to_numpy()[m] == verdad[m]).mean()):>8.0%}")

Ahí está. En los mensajes **de frontera** el modelo se queda en el 56 %, que es poco más
que tirar una moneda, mientras **la persona acierta el 91 %**.

Son mensajes como estos:

In [ ]:
import textwrap

# Buscamos una frase que aparezca con LAS DOS etiquetas. Si existe, la demostración ya
# no depende de que nos creamos nada: el mismo texto es una cosa o la otra.
frontera_prueba = buzon[buzon["frontera"] == 1].copy()


def sin_producto(texto):
    """Quita el nombre del producto para poder comparar dos mensajes de la misma
    plantilla. Es un apaño para la demostración, no un preprocesado del modelo."""
    return " ".join(texto.split()[:6])


frontera_prueba["inicio"] = frontera_prueba["texto"].map(sin_producto)
pares = (frontera_prueba.groupby("inicio")["categoria"].nunique()
         .sort_values(ascending=False))
clave = pares.index[0]
ejemplos = frontera_prueba[frontera_prueba["inicio"] == clave]

for categoria in ["devolucion", "garantia"]:
    fila = ejemplos[ejemplos["categoria"] == categoria].iloc[0]
    print(f"[{categoria:^10}]  días desde la última compra: {fila['dias_desde_compra']:.0f}")
    print(textwrap.fill(fila["texto"], 96, initial_indent="  ", subsequent_indent="  "))
    print()

Ahí está la demostración: **el mismo arranque de mensaje aparece con las dos
etiquetas**. Lo que decide si es una **devolución** o una **garantía** es **cuánto hace
que se compró el aparato**: hasta catorce días naturales
hay derecho de desistimiento (artículo 102 del texto refundido de la Ley General para la
Defensa de los Consumidores y Usuarios); pasados esos catorce días lo que queda es la
garantía. El cliente no lo sabe y no lo dice.

La persona acierta porque, sin pensarlo, **abre la ficha del cliente y mira la fecha**.
Al modelo nunca se la hemos dado.

## 5. Modelo 3: el mismo modelo, con el contexto que la empresa ya tenía

Tres columnas más. Ni una línea de modelo nuevo, ni una biblioteca nueva, ni un minuto
más de entrenamiento.

> **Animación interactiva:** [El dato que le falta](https://rafasalaesteve.github.io/IABD-PIA/UD7_automatizacion_negocio/animaciones/02_el_dato_que_le_falta.html). Permite cambiar entre el modelo solo con el texto, el modelo con los días desde la compra y la persona sobre los 54 mensajes de frontera, y leer cualquiera de ellos para ver que el texto no lleva la fecha.

In [ ]:
def contexto(datos):
    """Tres columnas que salen del sistema de pedidos, no del mensaje:
    los días desde la última compra, si caen dentro del plazo de desistimiento,
    y si no hay ninguna compra detrás (que es señal de publicidad o de preventa)."""
    dias = datos["dias_desde_compra"]
    return np.c_[dias.fillna(-1) / 100.0,
                 (dias <= 14).fillna(False).astype(int),
                 dias.isna().astype(int)]


X_entrena_ctx = hstack([X_entrena, csr_matrix(contexto(entrena))]).tocsr()
X_prueba_ctx = hstack([X_prueba, csr_matrix(contexto(prueba))]).tocsr()

modelo_ctx = LogisticRegression(max_iter=3000, C=5).fit(X_entrena_ctx, entrena["categoria"])

inicio = time.perf_counter()
pred_ctx = modelo_ctx.predict(X_prueba_ctx)
segundos_ctx = time.perf_counter() - inicio

anota("modelo, texto + contexto", pred_ctx, segundos_ctx)

frontera = grupos["de frontera"].to_numpy()
print()
print(f"En los mensajes de frontera: {float((pred_texto[frontera] == verdad[frontera]).mean()):.0%}"
      f"  ->  {float((pred_ctx[frontera] == verdad[frontera]).mean()):.0%}")
print(f"Ahorro respecto del modelo de solo texto: "
      f"{resultados['modelo, solo texto']['coste_errores'] - resultados['modelo, texto + contexto']['coste_errores']:,.0f} EUR/año")

> ## La lección central de la unidad
>
> **Una automatización no suele fallar por el modelo. Falla porque le falta un dato que
> la empresa ya tiene.**
>
> Entre el modelo anterior y este hay casi cuatro puntos de exactitud y más de seis mil
> euros al año, y en medio no hay ningún avance técnico: hay tres columnas que estaban
> en el sistema de pedidos desde el primer día.
>
> Antes de cambiar de modelo, de pedir una tarjeta gráfica o de contratar un servicio de
> inteligencia artificial, hay una pregunta que sale casi siempre más barata:
> **¿qué mira la persona que hace hoy este trabajo?**

## 6. El techo: los mensajes que no resuelve nadie

Queda el otro grupo, y es igual de importante. Los mensajes **confusos** se quedan en el
57 % con contexto, en el 58 % sin él y en el 55 % con la persona. Nadie mejora ahí, y
conviene ver por qué.

In [ ]:
for _, fila in prueba[prueba["confuso"] == 1].head(5).iterrows():
    print(f"[{fila['categoria']:^10}] {fila['texto'][:115]}")
print()
print("«Me ha llegado roto»: ¿lo rompió el transporte o venía mal de fábrica?")
print("«¿Cuándo me devolvéis el dinero?»: ¿lo lleva administración o posventa?")
print()
print("La respuesta no está en el mensaje ni en la ficha del cliente: está en abrir la")
print("caja o en una decisión de organización que la empresa no ha tomado.")

In [ ]:
# El techo, calculado. Si en el 11 % de los mensajes nadie puede pasar de acertar
# algo más de la mitad, ningún sistema puede superar este número en este buzón.
proporcion_confusos = float((prueba["confuso"] == 1).mean())
mejor_en_confusos = 0.60      # la proporción mayoritaria declarada de cada pareja
techo = (1 - proporcion_confusos) * 1.0 + proporcion_confusos * mejor_en_confusos

print(f"Proporción de mensajes confusos:  {proporcion_confusos:.1%}")
print(f"Techo teórico del problema:       {techo:.1%}")
print(f"Nuestro mejor modelo:             {resultados['modelo, texto + contexto']['exactitud']:.1%}")
print()
print("El techo supone, además, que TODO lo demás se acierta, que no va a pasar.")
print("Prometer un 99 % en un proceso con un 11 % de casos indecidibles es prometer")
print("algo imposible, y quien lo prometa no ha mirado sus datos.")

> **Saber dónde está el techo y por qué existe es parte de evaluar un modelo de
> automatización.** La solución de los mensajes confusos no es informática: es decidir en
> la empresa qué equipo los lleva, o pedir el dato que falta antes de clasificar.
>
> Y **no hay técnica que arregle un proceso mal definido**. Automatizarlo lo que hace es
> congelar la indecisión en código.

## 7. El quinto modelo: el de lenguaje grande, y lo que no medimos de él

No lo ejecutamos en clase porque no tenemos con qué. En lugar de copiar una exactitud de
un artículo, vamos a hacer lo que hay que hacer cuando no se puede medir: **poner precio
a lo que sí se sabe y declarar el hueco**.

In [ ]:
# Tamaño real de nuestros mensajes, que es lo que se le mandaría.
palabras = prueba["texto"].str.split().str.len()
fichas_entrada = palabras.mean() * 1.4 + 250      # + la instrucción del sistema
fichas_salida = 8                                  # solo devuelve la categoría

# Orden de magnitud de las tarifas públicas de un modelo pequeño de gama alta,
# consultadas en septiembre de 2026. NO son un dato estable: hay que volver a mirarlas
# antes de usarlas en un presupuesto, y por eso van con fecha.
PRECIO_ENTRADA = 1.00      # dólares por millón de fichas
PRECIO_SALIDA = 5.00

coste_mensaje = (fichas_entrada * PRECIO_ENTRADA + fichas_salida * PRECIO_SALIDA) / 1e6

MANTENIMIENTO_MODELO_ANO = 1536.0     # 4 h/mes a 32 EUR/h, del cuaderno UD7_04

print(f"Fichas de entrada por mensaje, estimadas: {fichas_entrada:.0f}")
print(f"Coste por mensaje:      {coste_mensaje * 100:.4f} céntimos de dólar")
print(f"Coste al año (2.400):   {coste_mensaje * 2400:.2f} dólares")
print(f"Coste con 2.400 al DÍA: {coste_mensaje * 2400 * 261:,.0f} dólares al año")
print()
print(f"El clasificador entrenado tarda "
      f"{resultados['modelo, texto + contexto']['ms_por_mensaje']:.3f} ms por mensaje")
print("y no cuesta nada por llamada, pero sí cuesta mantenerlo.")
print()
volumen_igualdad = MANTENIMIENTO_MODELO_ANO / coste_mensaje
print(f"Harían falta {volumen_igualdad / 1e6:.1f} millones de mensajes al año para que")
print("las llamadas al modelo de lenguaje costaran lo mismo que MANTENER el modelo")
print(f"entrenado ({MANTENIMIENTO_MODELO_ANO:,.0f} EUR/año).")

| | Clasificador entrenado | Modelo de lenguaje |
|---|---|---|
| Exactitud en este buzón | **94,4 %, medida** | **No medida.** No lo sabemos |
| Coste por mensaje | Prácticamente cero | Fracción de céntimo |
| Latencia | Menos de un milisegundo | Segundos |
| Necesita historial | Sí, y lo hay | No |
| Quién decide su comportamiento | Nosotros | El proveedor, y cambia sin avisar |
| Categoría nueva | Reentrenar | Cambiar una frase de la instrucción |

Y aquí hay que corregir un tópico, porque el número lo desmiente: **«los modelos de
lenguaje son caros» no se sostiene a este volumen, ni de lejos**. Clasificar los 2.400
mensajes del año cuesta menos de un dólar, y harían falta **millones de mensajes al año**
para que las llamadas costaran lo que cuesta mantener el modelo entrenado.

Lo que decide entre los dos, entonces, no es el precio por llamada:

- **La latencia**, si el proceso tiene que contestar en línea mientras el cliente espera.
- **La previsibilidad**: el clasificador entrenado hace hoy lo mismo que hizo ayer.
- **Quién controla el comportamiento del sistema**, que es el riesgo de dependencia del
  bloque 6 de la UD6 en su forma más aguda: el precio, el modelo y su comportamiento los
  decide otro y cambian sin avisar.
- **Que el proceso tenga o no historial etiquetado.** Si no lo tiene, la comparación ni
  siquiera existe: el modelo entrenado no se puede hacer.

> **Cuidado con descartar un modelo por una razón que no has medido.** El coste por
> llamada es la objeción que más se repite y, en un proceso de este tamaño, es la que
> menos pesa.

> **Un hueco declarado vale más que un número copiado.** Si en la P7.1 o en PR7 aparece
> una exactitud de un modelo de lenguaje sobre este buzón, tiene que venir de una
> medición propia. Si no la hay, se dice que no la hay.

## 8. Lo que el modelo hereda de quien etiquetó

Hasta aquí hemos entrenado contra `categoria`, que es la verdad. **En una empresa real
esa columna no existe.** Lo que hay es `categoria_asignada`: lo que puso la persona, con
su 13,2 % de errores dentro.

Vamos a ver qué pasa si entrenamos con lo que de verdad tendríamos.

In [ ]:
modelo_heredado = LogisticRegression(max_iter=3000, C=5).fit(
    X_entrena_ctx, entrena["categoria_asignada"])
pred_heredado = modelo_heredado.predict(X_prueba_ctx)

es_garantia = (prueba["categoria"] == "garantia").to_numpy()
print(f"{'entrenado con':<26} {'exactitud':>10} {'garantías mal':>15} {'coste errores':>15}")
for nombre, pred in [("la verdad", pred_ctx), ("lo que puso la persona", pred_heredado)]:
    mal = int(((pred != verdad) & es_garantia).sum())
    print(f"{nombre:<26} {float((pred == verdad).mean()):>9.1%} "
          f"{mal:>11} de {int(es_garantia.sum())} "
          f"{coste_de_los_errores(prueba, pred):>13,.0f} EUR")

Entrenar con las etiquetas de la persona cuesta algo más de dos puntos de exactitud y
**duplica las garantías mal encaminadas**, porque el modelo aprende precisamente la
confusión que la persona comete. El coste de los errores casi se dobla.

> **El sesgo del histórico no es una idea abstracta: es este número.** Y la única salida
> es **auditar una muestra a mano** —doscientos casos revisados por alguien que sepa— y
> eso son horas que tienen que estar en el presupuesto desde el principio.
>
> Un proyecto de automatización sin una partida de auditoría del histórico está mal
> presupuestado.

## 9. La partición aleatoria: lo que este conjunto NO puede demostrar

En la UD5, partir al azar unos datos con tiempo dentro mejoraba el resultado y la mejora
era falsa. Aquí vamos a hacer la misma comprobación, y el resultado **no es el que
esperábamos**. Merece la pena verlo y decirlo tal cual.

In [ ]:
from sklearn.model_selection import train_test_split

al_azar_ent, al_azar_pru = train_test_split(buzon, test_size=len(prueba),
                                            random_state=SEMILLA)
vec_azar = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
Xa = hstack([vec_azar.fit_transform(al_azar_ent["texto"]),
             csr_matrix(contexto(al_azar_ent))]).tocsr()
Xb = hstack([vec_azar.transform(al_azar_pru["texto"]),
             csr_matrix(contexto(al_azar_pru))]).tocsr()
pred_azar = LogisticRegression(max_iter=3000, C=5).fit(Xa, al_azar_ent["categoria"]).predict(Xb)

print(f"Partición temporal (honesta): {resultados['modelo, texto + contexto']['exactitud']:.1%}")
print(f"Partición aleatoria (trampa): {float((pred_azar == al_azar_pru['categoria']).mean()):.1%}")
print()
print("Aquí la trampa NO sube el resultado: lo baja un poco, y es ruido de haber")
print("evaluado sobre otros 627 mensajes.")

> **Y esto hay que decirlo tal cual, no taparlo.** En este conjunto la partición
> aleatoria no infla nada, porque el buzón es sintético y **no tiene deriva dentro**: los
> mensajes de diciembre se generan igual que los de enero. Es una limitación del material
> de clase, no un descubrimiento sobre el mundo.
>
> Lo que sigue siendo cierto, y es lo que hay que llevarse, es **por qué** se parte por
> tiempo: no para que el número baje, sino porque es la única partición que contesta a la
> pregunta que importa, que es **cómo se va a comportar el mes que viene**. En un buzón
> real, con campañas, catálogo cambiante y clientes nuevos, la diferencia aparece.
>
> Si en tu entrega escribes que la partición temporal «da un resultado peor pero más
> honesto», mídelo antes: aquí no lo da, y afirmarlo sería repetir lo que se espera oír
> en lugar de lo que se ha medido. Es justo lo que esta unidad no quiere.

## 10. La tabla que hay que llevarse

In [ ]:
tabla = pd.DataFrame([
    {"modelo": nombre,
     "exactitud": r["exactitud"],
     "errores al año": round((1 - r["exactitud"]) * len(buzon)),
     "coste de los errores": round(r["coste_errores"])}
    for nombre, r in resultados.items()])
tabla.loc[len(tabla)] = {
    "modelo": "la persona",
    "exactitud": float((prueba["categoria_asignada"] == prueba["categoria"]).mean()),
    "errores al año": round((1 - float((prueba["categoria_asignada"] == prueba["categoria"]).mean())) * len(buzon)),
    "coste de los errores": round(coste_de_los_errores(
        prueba, prueba["categoria_asignada"].to_numpy())),
}
tabla["exactitud"] = (tabla["exactitud"] * 100).round(1)
print(tabla.to_string(index=False))

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5))

nombres = list(resultados) + ["la persona"]
exactitudes = [resultados[n]["exactitud"] for n in resultados] + \
              [float((prueba["categoria_asignada"] == prueba["categoria"]).mean())]
colores = ["#9AA5B1", "#4878A8", "#4878A8", "#3B7A57", "#B4453C"]

ejes[0].barh(range(len(nombres)), exactitudes, color=colores)
ejes[0].set_yticks(range(len(nombres)), nombres)
ejes[0].axvline(exactitudes[0], color="#9AA5B1", linestyle="--",
                label="punto de referencia: clase mayoritaria")
ejes[0].axvline(exactitudes[-1], color="#B4453C", linestyle="--",
                label="punto de referencia: la persona")
ejes[0].set_xlim(0, 1)
ejes[0].set_xlabel("exactitud sobre los 3 últimos meses")
ejes[0].set_title("Los dos puntos de referencia, siempre a la vista")
ejes[0].legend(loc="lower right", fontsize=8)
ejes[0].invert_yaxis()

matriz = confusion_matrix(verdad, pred_ctx, labels=sorted(set(verdad)))
im = ejes[1].imshow(matriz, cmap="Blues")
etiquetas = sorted(set(verdad))
ejes[1].set_xticks(range(len(etiquetas)), etiquetas, rotation=45, ha="right")
ejes[1].set_yticks(range(len(etiquetas)), etiquetas)
ejes[1].set_xlabel("predicho")
ejes[1].set_ylabel("real")
ejes[1].set_title("Dónde falla el mejor modelo")
for i in range(len(etiquetas)):
    for j in range(len(etiquetas)):
        if matriz[i, j]:
            ejes[1].text(j, i, matriz[i, j], ha="center", va="center",
                         color="white" if matriz[i, j] > matriz.max() / 2 else "black",
                         fontsize=8)

plt.tight_layout()
plt.show()

Cinco frases para llevarse:

1. **Las reglas llegan al 72,7 % con dos horas de trabajo.** Ése es el punto de
   referencia que hay que batir, no el azar.
2. **El orden de las reglas vale más de veinte puntos.** Un sistema de reglas es una
   lista ordenada, y ese orden es conocimiento del negocio.
3. **El modelo bate a la persona por consistencia, no por inteligencia.**
4. **El salto que más vale no es técnico:** darle al modelo un dato que la empresa ya
   tenía sube los mensajes de frontera del 56 % al 96 %.
5. **Hay un techo, y está en los casos que no resuelve nadie.** Prometer por encima del
   techo es prometer algo imposible.

Lo que falta: una exactitud todavía no es una decisión. El cuaderno `UD7_03` pone precio
a cada error y calcula **hasta dónde conviene automatizar**, que es otra pregunta.